In [ ]:
import pandas as pd

In [ ]:
# Load the Iris dataset from the local CSV file.
df = pd.read_csv("iris.csv")

In [ ]:
print(f"Loaded {len(df)} rows and {len(df.columns)} columns")
df.head()

In [ ]:
# Basic dataset statistics
print("Shape:", df.shape)
print("\nMissing values per column:")
print(df.isna().sum())

print("\nNumerical summary:")
print(df.describe())

print("\nSpecies counts:")
print(df["Species"].value_counts())

In [ ]:
import matplotlib.pyplot as plt

# Scatter plot of petal measurements colored by species.
for species, group in df.groupby("Species"):
    plt.scatter(group["PetalLengthCm"], group["PetalWidthCm"], label=species, alpha=0.8)

plt.title("Iris Petal Length vs Petal Width")
plt.xlabel("Petal Length (cm)")
plt.ylabel("Petal Width (cm)")
plt.legend(title="Species")
plt.grid(alpha=0.2)
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split

feature_cols = ["SepalLengthCm", "SepalWidthCm", "PetalLengthCm", "PetalWidthCm"]
X = df[feature_cols]
y = df["Species"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=7, stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

In [ ]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(random_state=7, max_depth=3)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("Model trained.")

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print("\nClassification report:")
print(classification_report(y_test, y_pred))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
from sklearn.linear_model import LogisticRegression

logreg_model = LogisticRegression(max_iter=200, random_state=7)
logreg_model.fit(X_train, y_train)

logreg_pred = logreg_model.predict(X_test)

tree_acc = accuracy_score(y_test, y_pred)
logreg_acc = accuracy_score(y_test, logreg_pred)

comparison = pd.DataFrame(
    {
        "Model": ["Decision Tree", "Logistic Regression"],
        "Accuracy": [tree_acc, logreg_acc],
    }
)

print("Model accuracy comparison:")
comparison

print("\nLogistic Regression classification report:")
print(classification_report(y_test, logreg_pred))

In [ ]:
prediction_comparison = pd.DataFrame(
    {
        "Actual": y_test.reset_index(drop=True),
        "Decision Tree": pd.Series(y_pred),
        "Logistic Regression": pd.Series(logreg_pred),
    }
)

prediction_comparison["Same"] = (
    prediction_comparison["Decision Tree"] == prediction_comparison["Logistic Regression"]
)

print("Per-row prediction comparison:")
prediction_comparison

print("\nRows where the models disagree:", (~prediction_comparison["Same"]).sum())

In [ ]:
from sklearn.ensemble import VotingClassifier

ensemble_model = VotingClassifier(
    estimators=[
        ("tree", DecisionTreeClassifier(random_state=7, max_depth=3)),
        ("logreg", LogisticRegression(max_iter=200, random_state=7)),
    ],
    voting="soft",
)
ensemble_model.fit(X_train, y_train)
ensemble_pred = ensemble_model.predict(X_test)
ensemble_acc = accuracy_score(y_test, ensemble_pred)

print(f"Ensemble accuracy: {ensemble_acc:.3f}")
print("\nEnsemble classification report:")
print(classification_report(y_test, ensemble_pred))